# Retail Intelligence Platform - SQL Customer Analysis

This notebook builds the **customer analytics layer** for the Retail Intelligence Platform using PostgreSQL.

## Objectives
- analyze customer base and repeat purchase behavior
- calculate repeat customers and repeat customer rate
- calculate average orders per customer
- calculate revenue per customer
- analyze customer distribution by state and city
- export clean customer analysis outputs to `sql/outputs/`

## PostgreSQL tables used
- `olist_orders`
- `olist_order_items`
- `olist_customers`

## Reusable view used
- `vw_sales_fact`

In [37]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

# --------------------------------------------------
# PostgreSQL connection
# --------------------------------------------------
DB_USER = "postgres"
DB_PASSWORD = quote_plus("1234")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "retail_intelligence"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("PostgreSQL engine created successfully.")

PostgreSQL engine created successfully.


## 1. Quick connection and view validation
We first confirm that `vw_sales_fact` exists and can be queried successfully.

In [38]:
test_query = """
SELECT COUNT(*) AS total_rows
FROM vw_sales_fact;
"""

df_test = pd.read_sql(test_query, engine)
df_test

,total_rows
0,99441


## 2. Customer-level summary base

This section creates a reusable **customer summary view** with:
- first order date
- last order date
- total orders
- total revenue
- average order revenue
- customer tenure in days

In [39]:
create_customer_view_sql = """
DROP VIEW IF EXISTS vw_customer_order_summary;

CREATE VIEW vw_customer_order_summary AS
WITH delivered_orders AS (
    SELECT
        o.order_id,
        o.customer_id,
        c.customer_unique_id,
        o.order_purchase_timestamp::date AS order_date
    FROM olist_orders o
    LEFT JOIN olist_customers c
        ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
),

order_revenue AS (
    SELECT
        oi.order_id,
        SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)) AS order_revenue
    FROM olist_order_items oi
    GROUP BY oi.order_id
)

SELECT
    d.customer_unique_id,
    MIN(d.order_date) AS first_order_date,
    MAX(d.order_date) AS last_order_date,
    COUNT(DISTINCT d.order_id) AS total_orders,
    ROUND(SUM(COALESCE(r.order_revenue, 0))::numeric, 2) AS total_spent,
    ROUND(AVG(COALESCE(r.order_revenue, 0))::numeric, 2) AS avg_order_revenue,
    MAX(d.order_date) - MIN(d.order_date) AS customer_tenure_days,
    CASE
        WHEN COUNT(DISTINCT d.order_id) > 1 THEN TRUE
        ELSE FALSE
    END AS is_repeat_customer
FROM delivered_orders d
LEFT JOIN order_revenue r
    ON d.order_id = r.order_id
GROUP BY d.customer_unique_id;
"""

with engine.begin() as conn:
    conn.exec_driver_sql(create_customer_view_sql)

print("vw_customer_order_summary created successfully.")

vw_customer_order_summary created successfully.


## 3. Preview customer summary view

In [40]:
customer_summary_preview_query = """
SELECT *
FROM vw_customer_order_summary
LIMIT 10;
"""

df_customer_summary_preview = pd.read_sql(customer_summary_preview_query, engine)
df_customer_summary_preview

,customer_unique_id,first_order_date,last_order_date,total_orders,total_spent,avg_order_revenue,customer_tenure_days,is_repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-10,1,141.90,141.90,0,False
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07,2018-05-07,1,27.19,27.19,0,False
2,0000f46a3911fa3c0805444483337064,2017-03-10,2017-03-10,1,86.22,86.22,0,False
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12,2017-10-12,1,43.62,43.62,0,False
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14,2017-11-14,1,196.89,196.89,0,False
5,0004bd2a26a76fe21f786e4fbd80607f,2018-04-05,2018-04-05,1,166.98,166.98,0,False
6,00050ab1314c0e55a6ca13cf7181fecf,2018-04-20,2018-04-20,1,35.38,35.38,0,False
7,00053a61a98854899e70ed204dd4bafe,2018-02-28,2018-02-28,1,419.18,419.18,0,False
8,0005e1862207bf6ccc02e4228effd9a0,2017-03-04,2017-03-04,1,150.12,150.12,0,False
9,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-12,2018-03-12,1,129.76,129.76,0,False


## 4. Customer KPI Summary

This section calculates the main customer KPIs:
- total customers
- repeat customers
- one-time customers
- repeat customer rate
- average orders per customer
- average revenue per customer
- average customer tenure

In [41]:
customer_kpi_query = """
SELECT
    COUNT(*) AS total_customers,
    COUNT(CASE WHEN is_repeat_customer = TRUE THEN 1 END) AS repeat_customers,
    COUNT(CASE WHEN is_repeat_customer = FALSE THEN 1 END) AS one_time_customers,
    ROUND(
        100.0 * COUNT(CASE WHEN is_repeat_customer = TRUE THEN 1 END) / NULLIF(COUNT(*), 0),
        4
    ) AS repeat_customer_rate_pct,
    ROUND(AVG(total_orders)::numeric, 4) AS avg_orders_per_customer,
    ROUND(AVG(total_spent)::numeric, 4) AS avg_customer_spend,
    ROUND(AVG(customer_tenure_days)::numeric, 2) AS avg_customer_tenure_days
FROM vw_customer_order_summary;
"""

df_customer_kpis = pd.read_sql(customer_kpi_query, engine)
df_customer_kpis

,total_customers,repeat_customers,one_time_customers,repeat_customer_rate_pct,avg_orders_per_customer,avg_customer_spend,avg_customer_tenure_days
0,93358,2801,90557,3.0003,1.0334,165.1682,2.65


## 5. Customer order frequency distribution
This shows how many customers placed 1 order, 2 orders, 3 orders, and so on.

In [42]:
customer_order_frequency_query = """
SELECT
    total_orders,
    COUNT(*) AS customer_count,
    ROUND(AVG(total_spent)::numeric, 2) AS avg_customer_spend
FROM vw_customer_order_summary
GROUP BY total_orders
ORDER BY total_orders;
"""

df_customer_order_frequency = pd.read_sql(customer_order_frequency_query, engine)
df_customer_order_frequency

,total_orders,customer_count,avg_customer_spend
0,1,90557,160.73
1,2,2573,291.03
2,3,181,432.78
3,4,28,788.79
4,5,9,734.18
5,6,5,691.11
6,7,3,946.85
7,9,1,1172.67
8,15,1,879.27


## 6. Top customers by revenue
This section identifies the highest-value customers based on total revenue.

In [43]:
top_customers_query = """
SELECT
    customer_unique_id,
    total_orders,
    ROUND(total_spent::numeric, 2) AS total_spent,
    ROUND(avg_order_revenue::numeric, 2) AS avg_order_revenue,
    customer_tenure_days,
    is_repeat_customer
FROM vw_customer_order_summary
ORDER BY total_spent DESC
LIMIT 20;
"""

df_top_customers = pd.read_sql(top_customers_query, engine)
df_top_customers

,customer_unique_id,total_orders,total_spent,avg_order_revenue,customer_tenure_days,is_repeat_customer
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,13664.08,0,False
1,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,3785.82,0,True
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,7274.88,0,False
3,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,6929.31,0,False
4,459bef486812aa25204be022145caa62,1,6922.21,6922.21,0,False
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66,6726.66,0,False
6,4007669dec559734d6f53e029e360987,1,6081.54,6081.54,0,False
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34,4764.34,0,False
8,48e1ac109decbb87765a3eade6854098,1,4681.78,4681.78,0,False
9,c8460e4251689ba205045f3ea17884a1,4,4655.88,1163.97,1,True


## 7. Customer state analysis
This section summarizes customer counts, orders, and revenue by state.

In [44]:
customer_state_query = """
SELECT
    customer_state,
    COUNT(DISTINCT customer_unique_id) AS total_customers,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(item_revenue)::numeric, 2) AS total_revenue,
    ROUND(AVG(item_revenue)::numeric, 2) AS avg_order_value
FROM vw_sales_fact
GROUP BY customer_state
ORDER BY total_revenue DESC;
"""

df_customer_state = pd.read_sql(customer_state_query, engine)
df_customer_state

,customer_state,total_customers,total_orders,total_revenue,avg_order_value
0,SP,40302,41746,5202955.05,124.63
1,RJ,12384,12852,1824092.67,141.93
2,MG,11259,11635,1585308.03,136.25
3,RS,5277,5466,750304.02,137.27
4,PR,4882,5045,683083.76,135.40
5,SC,3534,3637,520553.34,143.13
6,BA,3277,3380,511349.99,151.29
7,DF,2075,2140,302603.94,141.40
8,GO,1952,2020,294591.95,145.84
9,ES,1964,2033,275037.31,135.29


## 8. Top customer cities
This section identifies the customer cities contributing the most orders and revenue.

In [45]:
customer_city_query = """
SELECT
    customer_city,
    customer_state,
    COUNT(DISTINCT customer_unique_id) AS total_customers,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(item_revenue)::numeric, 2) AS total_revenue
FROM vw_sales_fact
GROUP BY customer_city, customer_state
ORDER BY total_revenue DESC
LIMIT 25;
"""

df_customer_city = pd.read_sql(customer_city_query, engine)
df_customer_city

,customer_city,customer_state,total_customers,total_orders,total_revenue
0,sao paulo,SP,14984,15540,1914924.54
1,rio de janeiro,RJ,6620,6882,992538.86
2,belo horizonte,MG,2672,2773,355611.13
3,brasilia,DF,2069,2131,301920.25
4,curitiba,PR,1465,1521,211738.06
5,porto alegre,RS,1326,1379,190562.08
6,campinas,SP,1398,1444,187844.53
7,salvador,BA,1209,1245,181104.42
8,guarulhos,SP,1153,1189,144268.39
9,niteroi,RJ,811,849,117907.12


## 9. First purchase month cohort base
This section creates a customer cohort base using the first purchase month.
It will be reused later in retention analysis.

In [46]:
customer_cohort_query = """
WITH customer_first_order AS (
    SELECT
        customer_unique_id,
        MIN(order_month) AS first_order_month
    FROM vw_sales_fact
    GROUP BY customer_unique_id
)
SELECT
    first_order_month,
    COUNT(*) AS customers_acquired
FROM customer_first_order
GROUP BY first_order_month
ORDER BY first_order_month;
"""

df_customer_cohort = pd.read_sql(customer_cohort_query, engine)
df_customer_cohort

,first_order_month,customers_acquired
0,2016-09-01,4
1,2016-10-01,321
2,2016-12-01,1
3,2017-01-01,764
4,2017-02-01,1752
5,2017-03-01,2636
6,2017-04-01,2352
7,2017-05-01,3596
8,2017-06-01,3139
9,2017-07-01,3894


## 10. Repeat customer segmentation
This section classifies customers into simple frequency-based segments.

In [47]:
customer_segment_query = """
SELECT
    CASE
        WHEN total_orders = 1 THEN 'One-time'
        WHEN total_orders BETWEEN 2 AND 3 THEN 'Repeat'
        WHEN total_orders BETWEEN 4 AND 6 THEN 'Loyal'
        ELSE 'High-value'
    END AS customer_segment,
    COUNT(*) AS customer_count,
    ROUND(AVG(total_spent)::numeric, 2) AS avg_customer_spend,
    ROUND(AVG(total_orders)::numeric, 2) AS avg_orders
FROM vw_customer_order_summary
GROUP BY
    CASE
        WHEN total_orders = 1 THEN 'One-time'
        WHEN total_orders BETWEEN 2 AND 3 THEN 'Repeat'
        WHEN total_orders BETWEEN 4 AND 6 THEN 'Loyal'
        ELSE 'High-value'
    END
ORDER BY customer_count DESC;
"""

df_customer_segments = pd.read_sql(customer_segment_query, engine)
df_customer_segments

,customer_segment,customer_count,avg_customer_spend,avg_orders
0,One-time,90557,160.73,1.00
1,Repeat,2754,300.34,2.07
2,Loyal,42,765.46,4.45
3,High-value,5,978.50,9.00


## 11. Export customer outputs
Save all customer analysis outputs into the project `sql/outputs/` folder.

In [48]:
OUTPUT_DIR = Path(r"C:\Users\divya\Retail-Intelligence-Platform\sql\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Main SQL customer outputs
df_customer_kpis.to_csv(OUTPUT_DIR / "customer_kpi_summary_sql.csv", index=False)
df_customer_order_frequency.to_csv(OUTPUT_DIR / "customer_order_frequency_sql.csv", index=False)
df_top_customers.to_csv(OUTPUT_DIR / "top_customers_sql.csv", index=False)
df_customer_state.to_csv(OUTPUT_DIR / "customer_state_summary_sql.csv", index=False)
df_customer_city.to_csv(OUTPUT_DIR / "customer_city_summary_sql.csv", index=False)
df_customer_cohort.to_csv(OUTPUT_DIR / "customer_cohort_summary_sql.csv", index=False)
df_customer_segments.to_csv(OUTPUT_DIR / "customer_segment_summary_sql.csv", index=False)

# Streamlit-style repeat purchase base
repeat_purchase_query = """
SELECT
    customer_unique_id,
    total_orders,
    total_spent,
    first_order_date,
    last_order_date,
    is_repeat_customer
FROM vw_customer_order_summary
ORDER BY total_spent DESC;
"""

df_repeat_purchase_summary = pd.read_sql(repeat_purchase_query, engine)
df_repeat_purchase_summary.to_csv(OUTPUT_DIR / "repeat_purchase_summary_sql.csv", index=False)

print("Customer SQL outputs exported successfully.")
print(f"Saved to: {OUTPUT_DIR}")

Customer SQL outputs exported successfully.
Saved to: C:\Users\divya\Retail-Intelligence-Platform\sql\outputs


## 12. Summary

This notebook created the SQL customer analytics layer for the Retail Intelligence Platform.

### Outputs generated
- `customer_summary.csv`
- `customer_order_frequency.csv`
- `top_customers.csv`
- `customer_state_summary.csv`
- `customer_city_summary.csv`
- `customer_cohort_summary.csv`
- `customer_segment_summary.csv`

### Next notebook
`03_sql_seller_analysis.ipynb`